# data cleaning

In [64]:
%pip install pandas numpy matplotlib seaborn scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [65]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import sklearn
from sklearn.preprocessing import StandardScaler

# from thefuzz import process
from difflib import get_close_matches

In [66]:
# import 'Pandas' 
import pandas as pd 

# import 'Numpy' 
import numpy as np

# import subpackage of Matplotlib
import matplotlib.pyplot as plt

# import 'Seaborn' 
import seaborn as sns

# to suppress warnings 
from warnings import filterwarnings
filterwarnings('ignore')

# display all columns of the dataframe
pd.options.display.max_columns = None

# display all rows of the dataframe
pd.options.display.max_rows = None
 
# to display the float values upto 6 decimal places     
pd.options.display.float_format = '{:.6f}'.format

# import train-test split 
from sklearn.model_selection import train_test_split

# # import various functions from statsmodels
# import statsmodels
import statsmodels.api as sm
# import statsmodels.stats.api as sms
# from statsmodels.graphics.gofplots import qqplot

# # import 'stats'
# from scipy import stats

# # 'metrics' from sklearn is used for evaluating the model performance
# from sklearn.metrics import mean_squared_error

# # import functions to perform feature selection
# from mlxtend.feature_selection import SequentialFeatureSelector as sfs
# #from sklearn.feature_selection import SelectFromModel
from sklearn.feature_selection import RFE

# import function to perform linear regression
from sklearn.linear_model import LinearRegression

# import functions to perform cross validation
from sklearn.model_selection import LeaveOneOut
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold

In [67]:
train = pd.read_csv('train_(2)_(1)_(1)_(1).csv')
test = pd.read_csv('test_(2)_(1)_(1)_(1).csv')
avgRent = pd.read_csv('avg_rent_(1)_(1)_(1)_(1).csv')
disCityCenter = pd.read_csv('dist_from_city_centre_(1)_(1)_(1)_(1).csv')
sampleSubmission = pd.read_csv('sample_submission_(3)_(1)_(1)_(1).csv')

In [68]:
test.head()

,ID,area_type,availability,location,size,society,total_sqft,bath,balcony
0,0,Super built-up Area,Ready To Move,Chamrajpet,2 BHK,NaN,650,1.000000,1.000000
1,1,Super built-up Area,Ready To Move,7th Phase JP Nagar,3 BHK,SrncyRe,1370,2.000000,1.000000
2,2,Super built-up Area,Ready To Move,Whitefield,3 BHK,AjhalNa,1725,3.000000,2.000000
3,3,Built-up Area,Ready To Move,Jalahalli,2 BHK,NaN,1000,2.000000,0.000000
4,4,Plot Area,Ready To Move,TC Palaya,1 Bedroom,NaN,1350,1.000000,0.000000


In [69]:
print(test.info())
# test.dropna(subset=['size', 'location', 'bath', 'balcony'], inplace=True)
# test['size'].fillna(test['size'].mode(), inplace=True)

test['size'].fillna('2 BHK', inplace=True)
test['bath'].fillna(test['bath'].median(), inplace=True)
test['balcony'].fillna(test['balcony'].median(), inplace=True)
print(test.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2664 entries, 0 to 2663
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   ID            2664 non-null   int64  
 1   area_type     2664 non-null   object 
 2   availability  2664 non-null   object 
 3   location      2664 non-null   object 
 4   size          2662 non-null   object 
 5   society       1590 non-null   object 
 6   total_sqft    2664 non-null   object 
 7   bath          2656 non-null   float64
 8   balcony       2559 non-null   float64
dtypes: float64(2), int64(1), object(6)
memory usage: 187.4+ KB
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2664 entries, 0 to 2663
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   ID            2664 non-null   int64  
 1   area_type     2664 non-null   object 
 2   availability  2664 non-null   object 
 3   location      2664 non-nul

In [70]:



print(train.info())
train.dropna(subset=['size', 'location', 'bath', 'balcony'], inplace=True)
print(train.info())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10656 entries, 0 to 10655
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   ID            10656 non-null  int64  
 1   area_type     10656 non-null  object 
 2   availability  10656 non-null  object 
 3   location      10655 non-null  object 
 4   size          10642 non-null  object 
 5   society       6228 non-null   object 
 6   total_sqft    10656 non-null  object 
 7   bath          10591 non-null  float64
 8   balcony       10152 non-null  float64
 9   price         10656 non-null  float64
dtypes: float64(3), int64(1), object(6)
memory usage: 832.6+ KB
None
<class 'pandas.core.frame.DataFrame'>
Index: 10151 entries, 0 to 10655
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   ID            10151 non-null  int64  
 1   area_type     10151 non-null  object 
 2   availability  10151 non-n

### cleaning total_sqft and making it numeric 
as it is expected to be in sq feet

then scaling it

In [71]:
def is_number(x):
    try:
        float(x)
        return True
    except ValueError:
        return False

def return_sqft(x):
    if is_number(x):
        return float(x)
    
    tokens = x.split('-')
    if len(tokens) == 2:
        return (float(tokens[0]) + float(tokens[1])) / 2
    if x.endswith('Sq. Meter') or x.endswith('Sq. Meter'):
        x = float(x.split('Sq')[0])
        return x*10.7639
    if x.endswith('Acres'):
        x = float(x.split('Acr')[0])
        return x*43560
    if x.endswith('Sq. Yards'):
        x = float(x.split('Sq')[0])
        return x*10.7639
    if x.endswith('Grounds'):
        x = float(x.split('Gro')[0])
        return x*2400
    if x.endswith('Cents'):
        x = float(x.split('Cen')[0])
        return x*435.6
    if x.endswith('Perch'):
        x = float(x.split('Per')[0])
        return x*272.25
    if x.endswith('Guntha'):
        x = float(x.split('Gun')[0])
        return x*1089
    
    print("Couldn't parse the value: ", x)
    return x



train['total_sqft'] = train.total_sqft.apply(lambda x: return_sqft(x))
train[train.total_sqft.apply(lambda x: is_number(x) == False)]


# train['price_per_sqft'] = train.price / train.total_sqft
# train.total_sqft.describe()


test['total_sqft'] = test.total_sqft.apply(lambda x: return_sqft(x))
test[test.total_sqft.apply(lambda x: is_number(x) == False)]


,ID,area_type,availability,location,size,society,total_sqft,bath,balcony


outlier treatment for total_sqft based on area_type


OUTLIERS HAVE NOT YET BEEN TREATED AND MUST BE TREATED FOR ALL CATEGORIES

In [72]:
# # FOR SOME REASON THE R^2 IS REDUCING
# # ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^ HENCE THE CODE IS COMMENTED OUT ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

# # TODO: outlier analysis to be done wrt area type 

# # plt.subplots = plt.subplots(figsize=(10,5))


# print(train.area_type.value_counts())
# plt.figure(figsize=(20,10))


# plt.subplot(2,4,1)
# plt.title('Price per sqft distribution for Super built-up  Area')
# sns.boxplot(train[train.area_type == 'Super built-up  Area'].price_per_sqft)
# plt.subplot(2,4,5)
# plt.title('Price per sqft distribution for Super built-up  Area')
# sns.histplot(train[train.area_type == 'Super built-up  Area'].price_per_sqft)



# plt.subplot(2,4,2)
# plt.title('Price per sqft distribution for Built-up  Area')
# sns.boxplot(train[train.area_type == 'Built-up  Area'].price_per_sqft)
# plt.subplot(2,4,6)
# plt.title('Price per sqft distribution for Built-up  Area')
# sns.histplot(train[train.area_type == 'Built-up  Area'].price_per_sqft)



# plt.subplot(2,4,3)
# plt.title('Price per sqft distribution for Plot  Area')
# sns.boxplot(train[train.area_type == 'Plot  Area'].price_per_sqft)
# plt.subplot(2,4,7)
# plt.title('Price per sqft distribution for Plot  Area')
# sns.histplot(train[train.area_type == 'Plot  Area'].price_per_sqft)



# plt.subplot(2,4,4)
# plt.title('Price per sqft distribution for Carpet  Area')
# sns.boxplot(train[train.area_type == 'Carpet  Area'].price_per_sqft)
# plt.subplot(2,4,8)
# plt.title('Price per sqft distribution for Carpet  Area')
# sns.histplot(train[train.area_type == 'Carpet  Area'].price_per_sqft)



# plt.tight_layout()
# plt.show()

train['price_per_sqft'] = train.price / train.total_sqft
for i in train.area_type.unique():
    col = train[train.area_type == i].price_per_sqft
    iqr = (col.quantile(0.75) - col.quantile(0.25)) * (3/2)
    train.drop(train[(train.area_type == i) & (train.price_per_sqft > iqr)].index, inplace=True)



# for i in test.area_type.unique():
#     col = test[test.area_type == i].price_per_sqft
#     iqr = (col.quantile(0.75) - col.quantile(0.25)) * (3/2)
#     test.drop(test[(test.area_type == i) & (test.price_per_sqft > iqr)].index, inplace=True)

In [73]:
# print(train.price_per_sqft.describe())
# print(train[train.price_per_sqft > .6])
# train.drop(train[train.price_per_sqft > 0.6].index, inplace=True, axis=0)


In [74]:
# scaling data => total sq feet

# note that we must do outlier treatment before scaling
# min max scaling for total_sqft

# train.total_sqft = train.total_sqft.apply(lambda x: (x - train.total_sqft.min())/(train.total_sqft.max() - train.total_sqft.min()))

# all cols are being scaled later

successfully cleaned all non numeric data from total_sqft col

## handling size
- currently 1rk = 0
- 1bhk/1bedroom = 1
- 2bhk/2bedroom = 2 
and soo on

In [75]:
# print(train['size'].value_counts(dropna=False))
# train[train['size'].apply(lambda x: is_number(x) == False)]

def returnSize(x):
    try:
        if x == '1 RK':
            return 0
        else:
            return int(x.split(' ')[0])
    except:
        print("Couldn't parse size: ", x)


train['size'] = train['size'].apply(lambda x: returnSize(x))
test['size'] = test['size'].apply(lambda x: returnSize(x))

# train.info()



## avilibility


In [76]:
from datetime import datetime

# train.availability.value_counts()

def noOfDays(x):
    if x == 'Ready To Move':
        return 0
    tokens = x.split('-')
    if len(tokens) == 2:
        now = datetime.now()

        date = datetime.strptime(f'{x}-{now.year}', '%d-%b-%Y')

        if (date - now).days < 0:
            date = datetime.strptime(f'{x}-{now.year + 1}', '%d-%b-%Y')
            
        return (date - now).days
    else:
        print("Couldn't parse availability: ", x)
        return -1


train.availability = train.availability.apply(lambda x: noOfDays(x))
test.availability = test.availability.apply(lambda x: noOfDays(x))

# train.info()


Couldn't parse availability:  Immediate Possession
Couldn't parse availability:  Immediate Possession


## encoding area type

n-1 dummy encoding performed 

with value counts

- area_type
- Super built-up  Area    6745
- Built-up  Area          1845
- Plot  Area              1496
- Carpet  Area              65

model performance to be tested including only Super built-up  Area 

In [77]:
temp = True
for i in train.area_type.unique():
    if temp:
        temp = False
        continue
    train["area_type_" + i.split("  ")[0].replace(" ", "_").replace("-", "_")] = train.area_type.apply(lambda x: 1 if x==i else 0)

train.drop('area_type', axis=1, inplace=True)

In [78]:
temp = True
for i in test.area_type.unique():
    if temp:
        temp = False
        continue
    test["area_type_" + i.split("  ")[0].replace(" ", "_").replace("-", "_")] = test.area_type.apply(lambda x: 1 if x==i else 0)

test.drop('area_type', axis=1, inplace=True)

# using average rent and distance from city center

In [81]:

# # Function to find the best match and its score using thefuzz.process.extractOne
# def find_closest_match(target, choices):
#     # process.extractOne returns a tuple: (closest_match, score, index)
#     match, score, _ = process.extractOne(target, choices)
#     return match, score

# # Apply the function across rows
# # The result will be a new DataFrame or Series containing the match and score
# train[['avgRentPlace', 'avgRentScore']] = train['location'].apply(
#     lambda x: pd.Series(find_closest_match(x, avgRent['location']))
# )


# train['average_rent'] = train.apply(
#     lambda row: avgRent.loc[avgRent['location'] == row['avgRentPlace'], 'avg_2bhk_rent'].values[0],
#     axis=1
# )

# # train.drop(['avgRentPlace', 'avgRentScore'], axis=1, inplace=True)


# train[['distanceField', 'distanceScore']] = train['location'].apply(
#     lambda x: pd.Series(find_closest_match(x, disCityCenter['location']))
# )

# train["distance_from_city_center"] = train.apply(
#     lambda row: disCityCenter.loc[
#         disCityCenter["location"] == row["distanceField"], "dist_from_city"
#     ].values[0],
#     axis=1,
# )

# # train.drop(['distanceField', 'distanceScore'], axis=1, inplace=True)


In [97]:

%pip install geopy

  Using cached geopy-2.4.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached geographiclib-2.1-py3-none-any.whl.metadata (1.6 kB)
Using cached geopy-2.4.1-py3-none-any.whl (125 kB)
Using cached geographiclib-2.1-py3-none-any.whl (40 kB)
Note: you may need to restart the kernel to use updated packages.


In [99]:
# importing geopy library
from geopy.geocoders import Nominatim

# calling the Nominatim tool
loc = Nominatim(user_agent="GetLoc")

# entering the location name
getLoc = loc.geocode("Narayanapura on Hennur Main Road")

# printing address
print(getLoc.address)

# printing latitude and longitude
print("Latitude = ", getLoc.latitude, "\n")
print("Longitude = ", getLoc.longitude)

AttributeError: 'NoneType' object has no attribute 'address'

In [83]:

# I dont want to scale the target variable
y = train['price']

# standard scaling

In [84]:
scaler = StandardScaler()

numeric_cols = train.select_dtypes(include=[np.number]).columns
train[numeric_cols] = scaler.fit_transform(train[numeric_cols])

In [85]:
numeric_cols = test.select_dtypes(include=[np.number]).columns
test[numeric_cols] = scaler.fit_transform(test[numeric_cols])

In [86]:
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2664 entries, 0 to 2663
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   ID                  2664 non-null   float64
 1   availability        2664 non-null   float64
 2   location            2664 non-null   object 
 3   size                2664 non-null   float64
 4   society             1590 non-null   object 
 5   total_sqft          2664 non-null   float64
 6   bath                2664 non-null   float64
 7   balcony             2664 non-null   float64
 8   area_type_Built_up  2664 non-null   float64
 9   area_type_Plot      2664 non-null   float64
 10  area_type_Carpet    2664 non-null   float64
dtypes: float64(9), object(2)
memory usage: 229.1+ KB


In [87]:
test.head()

,ID,availability,location,size,society,total_sqft,bath,balcony,area_type_Built_up,area_type_Plot,area_type_Carpet
0,-1.731401,-0.425140,Chamrajpet,-0.661886,NaN,-0.196237,-1.311147,-0.774311,-0.475347,-0.404127,-0.082479
1,-1.730100,-0.425140,7th Phase JP Nagar,0.153308,SrncyRe,-0.057916,-0.539236,-0.774311,-0.475347,-0.404127,-0.082479
2,-1.728800,-0.425140,Whitefield,0.153308,AjhalNa,0.010284,0.232674,0.493522,-0.475347,-0.404127,-0.082479
3,-1.727500,-0.425140,Jalahalli,-0.661886,NaN,-0.128998,-0.539236,-2.042145,2.103726,-0.404127,-0.082479
4,-1.726199,-0.425140,TC Palaya,-1.477080,NaN,-0.061759,-1.311147,-2.042145,-0.475347,2.474469,-0.082479


In [88]:
train.info()


<class 'pandas.core.frame.DataFrame'>
Index: 2122 entries, 1 to 10651
Data columns (total 13 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   ID                        2122 non-null   float64
 1   availability              2122 non-null   float64
 2   location                  2122 non-null   object 
 3   size                      2122 non-null   float64
 4   society                   831 non-null    object 
 5   total_sqft                2122 non-null   float64
 6   bath                      2122 non-null   float64
 7   balcony                   2122 non-null   float64
 8   price                     2122 non-null   float64
 9   price_per_sqft            2122 non-null   float64
 10  area_type_Super_built_up  2122 non-null   float64
 11  area_type_Built_up        2122 non-null   float64
 12  area_type_Carpet          2122 non-null   float64
dtypes: float64(11), object(2)
memory usage: 232.1+ KB


In [89]:
test.head()

,ID,availability,location,size,society,total_sqft,bath,balcony,area_type_Built_up,area_type_Plot,area_type_Carpet
0,-1.731401,-0.425140,Chamrajpet,-0.661886,NaN,-0.196237,-1.311147,-0.774311,-0.475347,-0.404127,-0.082479
1,-1.730100,-0.425140,7th Phase JP Nagar,0.153308,SrncyRe,-0.057916,-0.539236,-0.774311,-0.475347,-0.404127,-0.082479
2,-1.728800,-0.425140,Whitefield,0.153308,AjhalNa,0.010284,0.232674,0.493522,-0.475347,-0.404127,-0.082479
3,-1.727500,-0.425140,Jalahalli,-0.661886,NaN,-0.128998,-0.539236,-2.042145,2.103726,-0.404127,-0.082479
4,-1.726199,-0.425140,TC Palaya,-1.477080,NaN,-0.061759,-1.311147,-2.042145,-0.475347,2.474469,-0.082479


In [90]:
train.drop(['price_per_sqft'], axis=1, inplace=True)

x = train.drop('price', axis=1)
# y = train['price']

x = x.select_dtypes(include= 'number')
x = sm.add_constant(x)

X_train, X_test, y_train, y_test = train_test_split(x, y, random_state=1, test_size = 0.2)


In [91]:
X_train.head()

,const,ID,availability,size,total_sqft,bath,balcony,area_type_Super_built_up,area_type_Built_up,area_type_Carpet
717,1.000000,-1.517631,-0.465037,-0.602064,-0.054573,-0.472621,0.712635,1.291968,-0.442907,-0.072186
7911,1.000000,0.824880,-0.465037,-0.602064,-0.059114,-0.472621,-0.451500,-0.774013,-0.442907,13.853126
1211,1.000000,-1.356775,-0.465037,1.404605,-0.043222,1.550720,-0.451500,-0.774013,-0.442907,-0.072186
3826,1.000000,-0.505278,0.802950,0.066826,-0.052019,0.201826,0.712635,1.291968,-0.442907,-0.072186
9341,1.000000,1.290517,-0.465037,0.066826,-0.052019,-0.472621,0.712635,-0.774013,2.257809,-0.072186


In [92]:
myModel = sm.OLS(y_train, X_train).fit()
myModel.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                  price   R-squared:                       0.389
Model:                            OLS   Adj. R-squared:                  0.386
Method:                 Least Squares   F-statistic:                     119.4
Date:                Fri, 16 Jan 2026   Prob (F-statistic):          1.43e-173
Time:                        21:32:04   Log-Likelihood:                -9538.7
No. Observations:                1697   AIC:                         1.910e+04
Df Residuals:                    1687   BIC:                         1.915e+04
Df Model:                           9                                         
Covariance Type:            nonrobust                                         
============================================================================================
                               coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------
const                       80.6764      1.628     49.570      0.000      77.484      83.869
ID                           0.6409      1.613      0.397      0.691      -2.524       3.805
availability                -2.5492      1.638     -1.556      0.120      -5.762       0.664
size                       -10.2411      3.719     -2.753      0.006     -17.536      -2.946
total_sqft                   1.1710      1.598      0.733      0.464      -1.964       4.306
bath                        31.2200      3.790      8.237      0.000      23.786      38.654
balcony                     14.0089      1.747      8.019      0.000      10.582      17.435
area_type_Super_built_up   -37.5803      1.993    -18.855      0.000     -41.489     -33.671
area_type_Built_up         -26.0012      1.842    -14.116      0.000     -29.614     -22.388
area_type_Carpet            -6.1927      1.626     -3.808      0.000      -9.383      -3.003
==============================================================================
Omnibus:                     1872.587   Durbin-Watson:                   1.927
Prob(Omnibus):                  0.000   Jarque-Bera (JB):           255056.891
Skew:                           5.268   Prob(JB):                         0.00
Kurtosis:                      62.128   Cond. No.                         4.74
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [93]:
test = test.select_dtypes(include= 'number')
test = sm.add_constant(test)

In [94]:
with open('submit.csv', 'w') as f:
    f.write('ID,price\n')
    predictions = myModel.predict(test)
    for idx, pred in enumerate(predictions):
        f.write(f'{idx},{pred}\n')

In [95]:
for i in (myModel.predict(test)):
    print(type(i))
    print(i)

<class 'float'>
64.30011327325843
<class 'float'>
80.21348381143184
<class 'float'>
122.15412315941113
<class 'float'>
-26.20283605248699
<class 'float'>
-19.798389064322855
<class 'float'>
63.392099367688274
<class 'float'>
122.1556570852745
<class 'float'>
79.07462013703645
<class 'float'>
88.43957886313022
<class 'float'>
164.16432657207335
<class 'float'>
80.2311076532563
<class 'float'>
95.17084480791122
<class 'float'>
88.46158419035075
<class 'float'>
72.67068422097971
<class 'float'>
104.340244379092
<class 'float'>
112.95891490714844
<class 'float'>
106.28362331821361
<class 'float'>
88.47699922494843
<class 'float'>
69.20119212896034
<class 'float'>
-8.393742062483057
<class 'float'>
88.52381664160426
<class 'float'>
106.30128795735604
<class 'float'>
88.50861141226827
<class 'float'>
122.13338133202105
<class 'float'>
122.1038450730418
<class 'float'>
119.71296712692241
<class 'float'>
70.15701607630633
<class 'float'>
3.625348760503123
<class 'float'>
39.49218168318569
<cla